# Extract data from CERCA raw files

In [79]:
from docx import Document
from PyPDF2 import PdfReader
import re
import pandas as pd
from google.cloud import bigquery
import requests
from tqdm import tqdm
import time
from df2gspread import gspread2df as g2d

In [80]:
interest_centers = ['CRM', 'ICRA', 'CTFC']
file_path = '../data/external/5_Bibliometria_SIRIS_031125/'

## Extract DOI list

**Each file has a different format so we go center by center**

### CRM

In [81]:
center_name = 'CRM/'
file_name = 'DOI_2021_2024'

In [82]:
reader = PdfReader(file_path + 'BM_' + center_name + file_name + '.pdf')

doi_pattern = r'10\.\d{4,9}/[\S]+'
df_CRM = pd.DataFrame()
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    df_tmp = pd.DataFrame(re.findall(doi_pattern, text), columns = ['DOI'])
    df_CRM = pd.concat((df_CRM, df_tmp), ignore_index = True)
df_CRM['Center'] = 'CRM'
df_CRM


,DOI,Center
0,10.1007/s40574-021-00294-5,CRM
1,10.1109/TCBB.2021.3101278,CRM
2,10.1016/j.ijheatmasstransfer.2020.120601,CRM
3,10.1016/j.aim.2021.107693,CRM
4,10.1134/S0081543821010193,CRM
...,...,...
395,10.1063/5.0234410,CRM
396,10.1088/1361-6544/ad8c1b,CRM
397,10.1007/s00222-024-01301-0,CRM
398,10.1038/s41467-024-53586-7,CRM


### ICRA

In [83]:
center_name = 'ICRA/'
file_name = 'DOIS_ICRA'

In [84]:
df_ICRA = pd.read_csv(file_path + 'BM_' + center_name + file_name + '.csv')
df_ICRA = df_ICRA[['ICRA']].dropna().rename(columns = {'ICRA' : 'DOI'})
df_ICRA['Center'] = 'ICRA'
df_ICRA

,DOI,Center
0,10.1007/698_2020_662,ICRA
1,10.1016/S0166-526X(21)00066-0,ICRA
2,10.7773/cm.v47i3.3212,ICRA
3,10.1016/j.mex.2021.101349,ICRA
4,10.1016/S0166-526X(21)00092-1,ICRA
...,...,...
549,10.1016/j.chemosphere.2024.143806,ICRA
550,10.1016/j.biortech.2024.131663,ICRA
551,10.3390/su162411277,ICRA
552,10.5194/hess-28-5353-2024,ICRA


### CTFC

**Missing when we do the analysis: we will do it appart as we did with ISGlobal and BETA**

In [85]:
df_centers = pd.concat([df_CRM, df_ICRA], ignore_index = True)
df_centers.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_segonaentrega.csv', index = False)
df_centers

,DOI,Center
0,10.1007/s40574-021-00294-5,CRM
1,10.1109/TCBB.2021.3101278,CRM
2,10.1016/j.ijheatmasstransfer.2020.120601,CRM
3,10.1016/j.aim.2021.107693,CRM
4,10.1134/S0081543821010193,CRM
...,...,...
939,10.1016/j.chemosphere.2024.143806,ICRA
940,10.1016/j.biortech.2024.131663,ICRA
941,10.3390/su162411277,ICRA
942,10.5194/hess-28-5353-2024,ICRA


In [86]:
df_centers.DOI.nunique()

933

## Check which publications are not in OA using DOI

In [87]:
PROJECT_ID = 'siris-datasets'
DATASET_ID = 'openalex'

def bg_query(query):
    client = bigquery.Client(project=PROJECT_ID)
    df = client.query(query)
    return df.to_dataframe()

In [88]:
in_query = str(tuple(df_centers.DOI.str.lower().tolist())) # IMPORTANT TO LOWER THEM!
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       wa.author_order,
       wa.author_position,
       wa.is_corresponding,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      WHERE ww.DOI IN {in_query}
      """

df_OA = bg_query(sql).dropna(subset = 'DOI').reset_index(drop = True)
df_OA

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE
0,10.1038/s41598-022-20957-3,Marta Itarte,24,middle,False,71999127,ES
1,10.1002/lno.11963,Anastasia Hiskia,55,middle,True,203474044,GR
2,10.3389/fmicb.2021.678057,None,<NA>,None,<NA>,<NA>,None
3,10.5194/gmd-15-4597-2022,Orlane Anneville,39,middle,False,4210088668,FR
4,10.5194/gmd-15-4597-2022,Orlane Anneville,39,middle,False,70900168,FR
...,...,...,...,...,...,...,...
8039,10.2166/9781789061154_0163,Jean‐Philippe Steyer,18,middle,False,4210112285,FR
8040,10.1021/acs.est.2c02896,Ivo Iavicoli,18,middle,False,71267560,IT
8041,10.3390/w13172352,Eric D. van Hullebusch,18,middle,False,1294671590,FR
8042,10.3390/w13172352,Eric D. van Hullebusch,18,middle,False,204730241,FR


In [12]:
(df_OA.DOI.nunique()) / df_centers.DOI.nunique()

0.9892818863879957

### Identify CERCA authors using OA (98% of the dataset)

- By affiliation ID
- By raw affiliations
  - Using parents affiliations
  - Manually checking above > 1 per raw affiliation
  - String search with keywords for = 1 doi per raw affilation

In [18]:
cerca_centers = {'CRM' : ['4210122226'], # NOT IN OA
                 'ICRA' : ['2799562678']}

interest_cerca = list(set(sum(list(cerca_centers.values()), [])))
interest_cerca = [int(x) for x in interest_cerca if x != '']

cerca_authors = df_OA[df_OA.institution_id.isin(interest_cerca)].drop_duplicates(['DOI'])

cerca_parents = {'CRM' : ['123044942'],
                 'ICRA' : ['251424209']}

parents_cerca = list(set(sum(list(cerca_parents.values()), [])))
parents_cerca = [int(x) for x in parents_cerca if x != '']

cerca_possible_authors = df_OA[(df_OA.institution_id.isin(parents_cerca)) & (~df_OA.display_name.isin(cerca_authors.display_name))].drop_duplicates(['DOI'])
cerca_possible_authors

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE
60,10.2166/wst.2024.069,Mar Satorras,16,middle,False,123044942,ES
580,10.1038/s43247-021-00192-w,Ferran Romero,33,middle,False,251424209,ES
835,10.1016/j.scitotenv.2020.142344,Ladislav Mandarić,17,middle,False,251424209,ES
872,10.1242/dev.201911,Hugo Cano‐Fernández,1,first,False,123044942,ES
873,10.1007/s10745-022-00325-5,Enrica Garau,1,first,True,251424209,ES
...,...,...,...,...,...,...,...
7468,10.1016/j.marpolbul.2023.115152,Elisa García,10,middle,False,251424209,ES
7470,10.1016/j.marpolbul.2023.114846,Elisa García,10,middle,False,251424209,ES
7780,10.1111/ele.14488,Anna M. Romaní,12,middle,False,251424209,ES
7832,10.1111/jbi.14052,Ferran Romero,13,middle,False,251424209,ES


In [19]:
in_query = str(tuple(cerca_possible_authors.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       war.raw_affiliation,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
      WHERE ww.DOI IN {in_query}
      """
df_possible_cerca = bg_query(sql).dropna(subset = 'DOI').drop_duplicates(['DOI', 'raw_affiliation']).reset_index(drop = True).reset_index(drop = True)
df_possible_cerca

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,raw_affiliation,institution_id,COUNTRY_CODE
0,10.1016/j.cej.2020.126482,Adrián Jaén-Gil,"Catalan Institute for Water Research (ICRA), C...",2799562678,ES
1,10.1016/j.cej.2020.126482,Adrián Jaén-Gil,"Universitat de Girona, Girona, Spain",2799562678,ES
2,10.1111/jbi.14052,Anna Freixa,Universitat de Girona (UdG) Girona Spain,2799562678,ES
3,10.1111/jbi.14052,Anna Freixa,Catalan Institute for Water Research (ICRA) Gi...,2799562678,ES
4,10.1016/j.coesh.2024.100547,Meritxell Valentí-Quiroga,"LEQUIA. Institute of the Environment, Universi...",251424209,ES
...,...,...,...,...,...
755,10.1111/2041-210x.14387,Pedro Puig,Centre de Recerca Matemàtica Universitat Autòn...,4210122226,ES
756,10.1016/j.chemosphere.2024.142934,Lúcia H.M.L.M. Santos,"University of Girona, Girona, Spain",251424209,ES
757,10.1016/j.envpol.2020.116313,Juan Antonio Campillo,"Spanish Institute of Oceanography (IEO), Marin...",4210091775,ES
758,10.1038/s43247-021-00192-w,Lyubomir Kenderov,Софийски университет = Sofia University (1504 ...,58918642,BG


In [20]:
grouped = df_possible_cerca[df_possible_cerca.COUNTRY_CODE == 'ES'].groupby('raw_affiliation').count().sort_values('DOI', ascending = False)[['DOI']]
to_check = grouped[grouped.DOI > 1]
print(df_possible_cerca[df_possible_cerca.raw_affiliation.isin(to_check.index)].DOI.nunique())
to_check.to_csv('to_check_segonaentrega.csv')
to_check
### WE MISS 876 DOIS BY FILTERING CHECKING THE THRESHOLD OF 1 AND MANUALLY CHECK THE MISSING

92


,DOI
raw_affiliation,
"University of Girona, Girona, Spain",15
"Catalan Institute for Water Research (ICRA), Girona, Spain",6
"Departament de Matemàtiques, Universitat Autònoma de Barcelona, 08193 Bellaterra, Barcelona, Catalonia, Spain",5
"Catalan Institute for Water Research (ICRA), 17003 Girona, Spain",5
"Universitat de Girona, Girona, Spain",5
"Group of Molecular Microbial Ecology, Institute of Aquatic Ecology, University of Girona, Girona, Spain",4
"Catalan Institute for Water Research (ICRA), Emili Grahit 101, 17003 Girona, Spain",3
"Catalan Institute for Water Research (ICRA), C. Emili Grahit 101, 17003 Girona, Spain",3
"University of Girona, Plaça de Sant Domènec 3, 17004, Girona, Spain",3


In [21]:
df_check_1 = g2d.download('1J7uywXX7fsxjNbUSi24uQJiW5bTjbKHba1RjqdYBwQA', '>1', col_names = True, row_names = False)

compute  = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check_1[df_check_1.CERCA == 'TRUE'].raw_affiliation)]

len(set(list(compute.DOI.unique()) + list(cerca_authors.DOI.unique()))) / df_centers.DOI.nunique()

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


0.8295819935691319

In [22]:
cerca_string = {'CRM' : ['crm', 'cerdanyola', 'bellaterra'],
                'ICRA' : ['icra', 'girona']}
cerca_string = list(set(sum(list(cerca_string.values()), [])))


df_tmp = grouped[(grouped.DOI == 1) & (~grouped.index.isin(df_check_1.raw_affiliation))].reset_index()
df_tmp.to_csv('to_check_segonaentrega_2.csv')
df_tmp

,raw_affiliation,DOI
0,"Catalan Institute for Water Research (ICRA), E...",1
1,"Mathematisches Institut , Georg-August Univer...",1
2,") Centre de Recerca Matemàtica, Edifici Cc, Ca...",1
3,") Departament de Matemàtiques, Facultat de Ciè...",1
4,") Departament de Matemàtiques, Universitat Pol...",1
...,...,...
401,BETA Technological Centre- University of Vic- ...,1
402,"Aquatic Ecology Group, University of Vic (UVic...",1
403,Aquatic Ecology Group University of Vic (UVic‐...,1
404,"Aigües de Vic S.A., Carrer de la Riera, 08500 ...",1


In [23]:
df_check_v2 = g2d.download('1J7uywXX7fsxjNbUSi24uQJiW5bTjbKHba1RjqdYBwQA', '=1', col_names = True, row_names = False)
df_check_v2

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


,raw_affiliation,DOI,CERCA
0,"Catalan Institute for Water Research (ICRA), E...",1,TRUE
1,"Mathematisches Institut , Georg-August Univers...",1,FALSE
2,") Centre de Recerca Matemàtica, Edifici Cc, Ca...",1,TRUE
3,") Departament de Matemàtiques, Facultat de Ciè...",1,TRUE
4,") Departament de Matemàtiques, Universitat Pol...",1,FALSE
...,...,...,...
401,BETA Technological Centre- University of Vic- ...,1,FALSE
402,"Aquatic Ecology Group, University of Vic (UVic...",1,FALSE
403,Aquatic Ecology Group University of Vic (UVic‐...,1,FALSE
404,"Aigües de Vic S.A., Carrer de la Riera, 08500 ...",1,FALSE


In [24]:
check_1 = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check_1[df_check_1.CERCA == 'TRUE'].raw_affiliation)]
check_2 = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check_v2[df_check_v2.CERCA == 'TRUE'].raw_affiliation)]

cerca_doi = list(set(
    list(cerca_authors.DOI.unique()) +
    list(check_1.DOI.unique()) +
    list(check_2.DOI.unique())
))
len(cerca_doi) / df_centers.DOI.nunique()

0.842443729903537

In [25]:
df_check = pd.concat((cerca_authors, check_1, check_2))
df_check

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,raw_affiliation
129,10.1038/s43247-021-00192-w,Anna Freixa,38,middle,False,2799562678,ES,NaN
144,10.1111/jbi.14052,Anna Freixa,28,middle,False,2799562678,ES,NaN
152,10.1016/j.ecolind.2022.108693,Dídac Jorda‐Capdevila,20,last,False,2799562678,ES,NaN
241,10.1007/978-3-031-12786-1_44,Mira Petrović,27,middle,False,2799562678,ES,NaN
431,10.1016/j.envint.2023.108288,Pablo Gago-Ferrero,35,middle,False,2799562678,ES,NaN
...,...,...,...,...,...,...,...,...
691,10.1016/j.jde.2024.03.030,Ernest Fontich,<NA>,NaN,<NA>,71999127,ES,"Departament de Matemàtiques i Informàtica, Uni..."
693,10.2140/apde.2021.14.717,Laura Prat,<NA>,NaN,<NA>,123044942,ES,"Departament de Matemàtiques, Universitat Autòn..."
694,10.3390/w16111496,Xavier García,<NA>,NaN,<NA>,2799562678,ES,Catalan Institute for Water Research (ICRA-CER...
726,10.1016/j.chemosphere.2024.143806,Oriol Gutiérrez,<NA>,NaN,<NA>,2799562678,ES,Catalan Institute for Water Research (ICRA- CE...


**In order to increase retrieval we study all the missing affiliations**

In [26]:
in_query = str(tuple(df_OA[~df_OA.DOI.isin(df_check.DOI)].drop_duplicates(['DOI']).DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       war.raw_affiliation,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
      WHERE ww.DOI IN {in_query}
      """
df_possible_cerca_2 = bg_query(sql).dropna(subset = 'DOI').drop_duplicates(['DOI', 'raw_affiliation']).reset_index(drop = True).reset_index(drop = True)
df_possible_cerca_2

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,raw_affiliation,institution_id,COUNTRY_CODE
0,10.23818/limn.43.07,Marc Sánchez-Morales,"Asociación Ibérica de Limnología, Madrid. Spain.",<NA>,None
1,10.1007/s10569-024-10203-5,Àngel Jorba,"Department de Matemàtiques i Informàtica, Univ...",71999127,ES
2,10.1007/s00332-024-10079-7,Xavier Jarque,None,<NA>,None
3,10.1007/s10240-024-00151-z,Marcel Guàrdia,None,<NA>,None
4,10.1007/978-3-031-63353-9_45,Matthew C. Ward,None,<NA>,None
...,...,...,...,...,...
256,10.1016/j.greeac.2022.100019,Muhammad Bilal,"School of Life Science and Food Engineering, H...",4210153869,CN
257,10.5565/publmat6612213,Clément Requilé,"Technische Universität Wien, Institute of Disc...",2799411422,US
258,10.1016/j.chemosphere.2024.142934,Alba Hernández,"Group of Mutagenesis, Department of Genetics a...",123044942,ES
259,10.1134/s1560354723520039,Eva Miranda,"Laboratory of Geometry and Dynamical Systems, ...",9617848,ES


In [29]:
df_possible_cerca_2[df_possible_cerca_2.COUNTRY_CODE == 'ES'].groupby('raw_affiliation').count().sort_values('DOI', ascending = False).to_csv('to_check_segonaentrega_3.csv')

In [30]:
df_check_v3 = g2d.download('1J7uywXX7fsxjNbUSi24uQJiW5bTjbKHba1RjqdYBwQA', 'Missing', col_names = True, row_names = False)

check_3 = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check_v3[df_check_v3.CERCA == 'TRUE'].raw_affiliation)]

cerca_doi = list(set(
    list(cerca_authors.DOI.unique()) +
    list(check_1.DOI.unique()) +
    list(check_2.DOI.unique()) + 
    list(check_3.DOI.unique()) 
))
len(cerca_doi) / df_centers.DOI.nunique()

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


0.842443729903537

In [31]:
df_check = pd.concat((cerca_authors, check_1, check_2, check_3))
df_check

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,raw_affiliation
129,10.1038/s43247-021-00192-w,Anna Freixa,38,middle,False,2799562678,ES,NaN
144,10.1111/jbi.14052,Anna Freixa,28,middle,False,2799562678,ES,NaN
152,10.1016/j.ecolind.2022.108693,Dídac Jorda‐Capdevila,20,last,False,2799562678,ES,NaN
241,10.1007/978-3-031-12786-1_44,Mira Petrović,27,middle,False,2799562678,ES,NaN
431,10.1016/j.envint.2023.108288,Pablo Gago-Ferrero,35,middle,False,2799562678,ES,NaN
...,...,...,...,...,...,...,...,...
499,10.3934/dcds.2021131,Lluı́s Alsedà,<NA>,NaN,<NA>,251424209,ES,Departament de Matemàtiques Universitat Autòno...
527,10.1007/s12346-020-00437-7,Armengol Gasull,<NA>,NaN,<NA>,123044942,ES,"Departament de Matemàtiques, Facultat de Ciènc..."
560,10.1016/j.jde.2024.05.037,Jordi Villadelprat,<NA>,NaN,<NA>,123044942,ES,"Departament de Matemàtiques, Edifici Cc, Unive..."
607,10.1103/physreve.107.064113,Joan del Castillo,<NA>,NaN,<NA>,123044942,ES,"Departament de Matemàtiques, Universitat Autòn..."


In [78]:
df_OA[~df_OA.DOI.isin(df_centers.DOI)]

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA
893,10.1029/2024jg008346,Xavier Peñarroya,1,first,True,4210101295,ES,False
902,10.1080/1943815x.2023.2264380,Josep Pueyo‐Ros,1,first,False,2799562678,ES,True
903,10.1080/1943815x.2023.2264380,Josep Pueyo‐Ros,1,first,False,251424209,ES,True
946,10.5817/cz.muni.eurocomb23-103,Miquel Ortega,1,first,False,9617848,ES,True
947,10.5817/cz.muni.eurocomb23-103,Miquel Ortega,1,first,False,4210122226,ES,True
...,...,...,...,...,...,...,...,...
7472,10.1117/1.nph.10.1.015006,Miriam de Nadal,10,middle,False,4210127641,ES,False
7497,10.1016/b978-0-323-85669-0.00066-0,Giancarlo Franzese,10,last,True,4401200369,None,False
7498,10.1016/b978-0-323-85669-0.00066-0,Giancarlo Franzese,10,last,True,71999127,ES,False
7590,10.1117/1.nph.10.1.015006,None,11,middle,False,4210127641,ES,True


In [89]:
# HI HA ERROR AMB L'ASSIGNACIÓ I PER AIXÒ SURT RAR! S'HA DE FER ALS POSSIBLE (PATENT I NO PARENT) I LLAVORS FER EL MERGE AMB EL OA QUE TÉ L'AUTHOR ORDER

df_OA['CERCA'] = (df_OA['display_name'].isin(df_check['display_name']) & df_OA['DOI'].isin(df_check['DOI']))
df_centers['DOI'] = df_centers['DOI'].str.lower()
df_final = df_OA.merge(df_centers, on = 'DOI', how = 'left').drop_duplicates(['DOI', 'display_name', 'institution_id']).reset_index(drop = True)
df_final.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA_segonaentrega.csv', index = False)
df_final

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center
0,10.1038/s41598-022-20957-3,Marta Itarte,24,middle,False,71999127,ES,False,ICRA
1,10.1002/lno.11963,Anastasia Hiskia,55,middle,True,203474044,GR,False,ICRA
2,10.3389/fmicb.2021.678057,None,<NA>,None,<NA>,<NA>,None,False,ICRA
3,10.5194/gmd-15-4597-2022,Orlane Anneville,39,middle,False,4210088668,FR,False,ICRA
4,10.5194/gmd-15-4597-2022,Orlane Anneville,39,middle,False,70900168,FR,False,ICRA
...,...,...,...,...,...,...,...,...,...
8035,10.2166/9781789061154_0163,Jean‐Philippe Steyer,18,middle,False,4210112285,FR,False,ICRA
8036,10.1021/acs.est.2c02896,Ivo Iavicoli,18,middle,False,71267560,IT,False,ICRA
8037,10.3390/w13172352,Eric D. van Hullebusch,18,middle,False,1294671590,FR,False,ICRA
8038,10.3390/w13172352,Eric D. van Hullebusch,18,middle,False,204730241,FR,False,ICRA


In [93]:
df_final.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CRM     387
ICRA    536
dtype: int64

In [91]:
df_centers.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CRM     390
ICRA    543
dtype: int64

In [92]:
df_final[df_final.CERCA == True].DOI.nunique() / df_centers.DOI.nunique()

0.842443729903537